# 00 · Set up the course data

This notebook copies the course data into your Databricks Free Edition workspace.
Run it **once**, from top to bottom, before the Spark DataFrame lessons.
It takes about 10–15 minutes, mostly for downloading and converting files.

**What you get.** A *volume* is a folder for files that lives inside Databricks.
This notebook creates one volume, `bdcc`, and fills it like this:

```text
/Volumes/workspace/default/bdcc/
├── raw/                          the original downloads (kept as a backup)
├── nyctaxi/
│   ├── green_2017/               NYC green taxi trips, Oct–Dec 2017, as Parquet
│   ├── green_tripdata_2017-10.csv
│   ├── green_tripdata_2017-11.csv
│   └── green_tripdata_2017-12.csv
├── gharchive/                    3 hours of GitHub events (JSON Lines, gzip)
├── lessons/                      2 tiny lesson files: 12 taxi trips, 8 GitHub events
└── gutenberg/1/1/1/              23 public-domain books (plain text)
```

**How to run it.**
1. At the top right, connect the notebook to **Serverless** compute.
2. Click **Run all**.
3. If a cell fails, read its message, fix the cause, and run that cell again.
   Every step is safe to run twice.

**Where the data comes from.** All three sources are public:
- NYC Taxi & Limousine Commission (TLC) trip records: green taxis, October to December 2017.
- GH Archive: a public log of activity on GitHub, one JSON object per line.
- Project Gutenberg: free e-books whose US copyright has expired.

## Step 1 · Create the volume

Databricks gives every file a three-part address: *catalog*, then *schema*, then *volume*.
(Here "schema" means a group of folders. In Session 2 the word means something else:
the list of a table's columns and their types.)
Free Edition already has the catalog `workspace` with the schema `default`.
The next two cells name our folder and create the volume `bdcc` inside them.

In [ ]:
# BASE is the folder that every course notebook reads from.
# Its parts are /Volumes/<catalog>/<schema>/<volume>.
BASE = "/Volumes/workspace/default/bdcc"
print(BASE)

In [ ]:
# Create the volume. "IF NOT EXISTS" makes this safe to run twice.
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.default.bdcc")

## Step 2 · Download the raw files

The cells below download each file into `BASE/raw/`.
A file that is already there is skipped, so running a cell again is cheap.
First we make the folders and a small helper function.

In [ ]:
import os
import urllib.request

# Make the three raw folders. Nothing happens if they already exist.
for sub in ["taxi", "gharchive", "gutenberg"]:
    dbutils.fs.mkdirs(f"{BASE}/raw/{sub}")

In [ ]:
def download(url, dest):
    """Save the file at url as dest. Skip it if dest already exists."""
    if os.path.exists(dest):
        print("already here:", dest)
        return True
    try:
        urllib.request.urlretrieve(url, dest)
        print("downloaded:  ", dest)
        return True
    except Exception as err:            # for example: no internet access
        if os.path.exists(dest):
            os.remove(dest)             # never keep a half-downloaded file
        print("FAILED:      ", url, "->", err)
        return False

In [ ]:
# Three months of green taxi trips (Parquet files from the TLC website)
TAXI_MONTHS = ["2017-10", "2017-11", "2017-12"]

# Three hours of GitHub events from 1 January 2015 (times are UTC)
GH_HOURS = ["2015-01-01-15", "2015-01-01-16", "2015-01-01-17"]

# The 23 books numbered 11100-11199 that still have a plain ?????.txt file
BOOK_IDS = ["11102", "11115", "11129", "11141", "11144", "11148", "11149",
            "11150", "11160", "11162", "11166", "11168", "11171", "11181",
            "11182", "11183", "11184", "11185", "11186", "11187", "11188",
            "11189", "11190"]

In [ ]:
# Taxi trips: about 40 MB in total
for month in TAXI_MONTHS:
    name = f"green_tripdata_{month}.parquet"
    download(f"https://d37ci6vzurychx.cloudfront.net/trip-data/{name}",
             f"{BASE}/raw/taxi/{name}")

In [ ]:
# GitHub events: about 12 MB in total
for hour in GH_HOURS:
    download(f"https://data.gharchive.org/{hour}.json.gz",
             f"{BASE}/raw/gharchive/{hour}.json.gz")

In [ ]:
# Books: about 3.5 MB in total. A book's folder comes from its digits:
# book 11102 lives at 1/1/1/0/11102/11102.txt (the 4th digit is "0").
MIRRORS = ["https://gutenberg.pglaf.org", "https://aleph.gutenberg.org"]
for book_id in BOOK_IDS:
    path = f"1/1/1/{book_id[3]}/{book_id}/{book_id}.txt"
    for mirror in MIRRORS:              # try the second mirror only if needed
        if download(f"{mirror}/{path}", f"{BASE}/raw/gutenberg/{book_id}.txt"):
            break

In [ ]:
# Which raw files are still missing?
expected = ([f"{BASE}/raw/taxi/green_tripdata_{m}.parquet" for m in TAXI_MONTHS]
            + [f"{BASE}/raw/gharchive/{h}.json.gz" for h in GH_HOURS]
            + [f"{BASE}/raw/gutenberg/{b}.txt" for b in BOOK_IDS])
missing = [p for p in expected if not os.path.exists(p)]
print(len(expected) - len(missing), "of", len(expected), "raw files are in place")
for p in missing:
    print("missing:", p)

### If a download failed: upload the files by hand

Some workspaces block downloads from the internet.
If the cell above reports missing files, you can upload them yourself:

1. On your own computer, download each missing file from the list below.
2. In Databricks, open **Catalog** in the left sidebar.
   Go to **workspace** → **default** → **Volumes** → **bdcc** → **raw**.
3. Open the matching folder (`taxi`, `gharchive` or `gutenberg`).
4. Click **Upload to this volume** and drop in the files.
   Keep the file names exactly as listed (each book is `<id>.txt`, for example `11102.txt`).
5. Run the "Which raw files are still missing?" cell again.
   When it says `29 of 29 raw files are in place`, continue with Step 3.

If `gutenberg.pglaf.org` does not answer, the same books are at `https://aleph.gutenberg.org/` with the same path.

- `raw/taxi/`: https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2017-10.parquet
- `raw/taxi/`: https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2017-11.parquet
- `raw/taxi/`: https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2017-12.parquet
- `raw/gharchive/`: https://data.gharchive.org/2015-01-01-15.json.gz
- `raw/gharchive/`: https://data.gharchive.org/2015-01-01-16.json.gz
- `raw/gharchive/`: https://data.gharchive.org/2015-01-01-17.json.gz
- `raw/gutenberg/11102.txt`: https://gutenberg.pglaf.org/1/1/1/0/11102/11102.txt
- `raw/gutenberg/11115.txt`: https://gutenberg.pglaf.org/1/1/1/1/11115/11115.txt
- `raw/gutenberg/11129.txt`: https://gutenberg.pglaf.org/1/1/1/2/11129/11129.txt
- `raw/gutenberg/11141.txt`: https://gutenberg.pglaf.org/1/1/1/4/11141/11141.txt
- `raw/gutenberg/11144.txt`: https://gutenberg.pglaf.org/1/1/1/4/11144/11144.txt
- `raw/gutenberg/11148.txt`: https://gutenberg.pglaf.org/1/1/1/4/11148/11148.txt
- `raw/gutenberg/11149.txt`: https://gutenberg.pglaf.org/1/1/1/4/11149/11149.txt
- `raw/gutenberg/11150.txt`: https://gutenberg.pglaf.org/1/1/1/5/11150/11150.txt
- `raw/gutenberg/11160.txt`: https://gutenberg.pglaf.org/1/1/1/6/11160/11160.txt
- `raw/gutenberg/11162.txt`: https://gutenberg.pglaf.org/1/1/1/6/11162/11162.txt
- `raw/gutenberg/11166.txt`: https://gutenberg.pglaf.org/1/1/1/6/11166/11166.txt
- `raw/gutenberg/11168.txt`: https://gutenberg.pglaf.org/1/1/1/6/11168/11168.txt
- `raw/gutenberg/11171.txt`: https://gutenberg.pglaf.org/1/1/1/7/11171/11171.txt
- `raw/gutenberg/11181.txt`: https://gutenberg.pglaf.org/1/1/1/8/11181/11181.txt
- `raw/gutenberg/11182.txt`: https://gutenberg.pglaf.org/1/1/1/8/11182/11182.txt
- `raw/gutenberg/11183.txt`: https://gutenberg.pglaf.org/1/1/1/8/11183/11183.txt
- `raw/gutenberg/11184.txt`: https://gutenberg.pglaf.org/1/1/1/8/11184/11184.txt
- `raw/gutenberg/11185.txt`: https://gutenberg.pglaf.org/1/1/1/8/11185/11185.txt
- `raw/gutenberg/11186.txt`: https://gutenberg.pglaf.org/1/1/1/8/11186/11186.txt
- `raw/gutenberg/11187.txt`: https://gutenberg.pglaf.org/1/1/1/8/11187/11187.txt
- `raw/gutenberg/11188.txt`: https://gutenberg.pglaf.org/1/1/1/8/11188/11188.txt
- `raw/gutenberg/11189.txt`: https://gutenberg.pglaf.org/1/1/1/8/11189/11189.txt
- `raw/gutenberg/11190.txt`: https://gutenberg.pglaf.org/1/1/1/9/11190/11190.txt

## Step 3 · Tidy the taxi trips into one Parquet folder

The three monthly files almost agree, but not quite:
- `trip_type` is a decimal number in October and December, and a whole number in November.
  Spark refuses to read files that disagree like this in one go.
- `ehail_fee` and `congestion_surcharge` are completely empty.
  (`congestion_surcharge` did not exist yet in 2017.)

So we read each month on its own and give every column one agreed type.
Then we drop `congestion_surcharge` and stack the months with `unionByName`.
The result is saved as **Parquet**, a file format that stores data column by column (Session 5).

In [ ]:
from pyspark.sql import functions as F

# Columns that hold whole numbers (IDs and counts)
INT_COLUMNS = ["VendorID", "RatecodeID", "PULocationID", "DOLocationID",
               "passenger_count", "payment_type", "trip_type"]

def read_month(month):
    """Read one raw month and give every column the agreed type."""
    df = spark.read.parquet(f"{BASE}/raw/taxi/green_tripdata_{month}.parquet")
    for name in INT_COLUMNS:
        df = df.withColumn(name, F.col(name).cast("int"))
    for name in ["lpep_pickup_datetime", "lpep_dropoff_datetime"]:
        df = df.withColumn(name, F.col(name).cast("timestamp"))
    df = df.withColumn("ehail_fee", F.col("ehail_fee").cast("double"))
    return df.drop("congestion_surcharge")

In [ ]:
# Stack the three months, then save them as one Parquet folder.
trips_all = read_month(TAXI_MONTHS[0])
for month in TAXI_MONTHS[1:]:
    trips_all = trips_all.unionByName(read_month(month))

trips_all.write.parquet(f"{BASE}/nyctaxi/green_2017", mode="overwrite")
print("Parquet rows:", spark.read.parquet(f"{BASE}/nyctaxi/green_2017").count())

## Step 4 · Write one CSV file per month

Session 1 reads CSV (comma-separated values) files, like the original course did.
Spark would save a CSV as a folder of many "part" files.
To get one ordinary file per month, we use the `pyarrow` library instead.
It reads the raw Parquet in chunks of 100,000 rows, so memory use stays small.

Each CSV has the same 19 columns as the Parquet folder.
Dates look like `2017-10-01 00:27:41`, and missing values are left empty.

In [ ]:
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.csv as pacsv
import pyarrow.parquet as pq

# The same 19 columns as the Parquet folder, in the same order
CSV_COLUMNS = ["VendorID", "lpep_pickup_datetime", "lpep_dropoff_datetime",
               "store_and_fwd_flag", "RatecodeID", "PULocationID", "DOLocationID",
               "passenger_count", "trip_distance", "fare_amount", "extra", "mta_tax",
               "tip_amount", "tolls_amount", "ehail_fee", "improvement_surcharge",
               "total_amount", "payment_type", "trip_type"]

In [ ]:
def tidy_batch(batch):
    """Give one chunk of rows the agreed types, ready for CSV."""
    cols = []
    for name in CSV_COLUMNS:
        col = batch.column(name)
        if name in ("lpep_pickup_datetime", "lpep_dropoff_datetime"):
            col = pc.strftime(col.cast(pa.timestamp("s")), format="%Y-%m-%d %H:%M:%S")
        elif name in INT_COLUMNS:
            col = col.cast(pa.int32())
        elif name == "ehail_fee":
            col = col.cast(pa.float64())
        cols.append(col)
    return pa.RecordBatch.from_arrays(cols, names=CSV_COLUMNS)

In [ ]:
def write_month_csv(month):
    """Write BASE/nyctaxi/green_tripdata_<month>.csv and return its row count."""
    source = pq.ParquetFile(f"{BASE}/raw/taxi/green_tripdata_{month}.parquet")
    rows = 0
    with open(f"{BASE}/nyctaxi/green_tripdata_{month}.csv", "wb") as f:
        f.write((",".join(CSV_COLUMNS) + "\n").encode())       # header line
        options = pacsv.WriteOptions(include_header=False, quoting_style="none")
        writer = None
        for batch in source.iter_batches(batch_size=100_000):
            batch = tidy_batch(batch)
            if writer is None:
                writer = pacsv.CSVWriter(f, batch.schema, write_options=options)
            writer.write_batch(batch)
            rows += batch.num_rows
        writer.close()
    return rows

In [ ]:
for month in TAXI_MONTHS:
    print(f"green_tripdata_{month}.csv:", write_month_csv(month), "rows")

In [ ]:
# Check: Spark reads the three CSV files back as one table.
csv_trips = spark.read.csv(f"{BASE}/nyctaxi/green_tripdata_2017-1*.csv", header=True)
print("CSV rows:", csv_trips.count())
csv_trips.show(3)

## Step 5 · Put the GitHub events and the books in place

The lessons read GitHub events from `BASE/gharchive/`.
They read books with the pattern `BASE/gutenberg/1/1/1/*/*/?????.txt`, like the original course.
`dbutils.fs.cp` copies files inside Databricks storage.

In [ ]:
# GitHub events: copy the whole folder
dbutils.fs.cp(f"{BASE}/raw/gharchive", f"{BASE}/gharchive", recurse=True)

# Books: copy each one into its numbered folder, e.g. 1/1/1/0/11102/11102.txt
for book_id in BOOK_IDS:
    folder = f"{BASE}/gutenberg/1/1/1/{book_id[3]}/{book_id}"
    dbutils.fs.cp(f"{BASE}/raw/gutenberg/{book_id}.txt", f"{folder}/{book_id}.txt")
print("copied", len(GH_HOURS), "event files and", len(BOOK_IDS), "books")

## Step 5b · Write the two lesson files

The lessons use two tiny files cut from the real data: 12 taxi trips (`lessons/taxi12.csv`)
and 8 GitHub events (`lessons/gh8.jsonl`). The next cell writes them into the volume,
so every lesson cell that reads them works in your notebook too.

In [ ]:
# The two lesson files, exactly as the lessons read them.
# taxi12.csv: 12 real green-taxi trips of 2 October 2017 (10 of the 19 columns).
TAXI12_CSV = """VendorID,lpep_pickup_datetime,store_and_fwd_flag,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,tip_amount,total_amount
2,2017-10-02 08:00:03,N,41,238,1,1.94,11.5,2.46,14.76
2,2017-10-02 08:00:06,N,97,137,1,5.08,22,2,24.8
1,2017-10-02 08:00:14,N,7,261,1,9.8,34.5,6,41.3
2,2017-10-02 08:00:15,N,7,179,1,0.37,4,0.96,5.76
2,2017-10-02 08:00:16,N,74,166,1,1.4,8.5,1.86,11.16
2,2017-10-02 08:00:30,N,33,140,1,8.1,29,4.47,34.27
2,2017-10-02 08:00:38,N,41,238,1,1.76,12,0,12.8
2,2017-10-02 08:00:51,N,7,260,1,1.51,8,0,8.8
2,2017-10-02 08:01:03,N,75,41,1,1.56,8,0,8.8
1,2017-10-02 08:01:08,N,74,75,1,1,6,0,6.8
2,2017-10-02 08:01:38,N,7,223,2,2.15,10.5,0,11.3
1,2017-10-02 10:47:29,Y,33,143,1,7.1,31,6.35,38.15
"""

# gh8.jsonl: 8 real GitHub events of 1 January 2015, trimmed to a few fields.
GH8_JSONL = """{"id": "2489651045", "type": "CreateEvent", "actor": {"login": "petroav"}, "repo": {"name": "petroav/6.828"}, "created_at": "2015-01-01T15:00:00Z"}
{"id": "2489651051", "type": "PushEvent", "actor": {"login": "rspt"}, "repo": {"name": "rspt/rspt-theme"}, "payload": {"commits": [{"author": {"name": "rspt"}, "message": "Fix main header height on mobile"}]}, "created_at": "2015-01-01T15:00:01Z"}
{"id": "2489651057", "type": "WatchEvent", "actor": {"login": "SametSisartenep"}, "repo": {"name": "visionmedia/debug"}, "created_at": "2015-01-01T15:00:03Z"}
{"id": "2489651078", "type": "WatchEvent", "actor": {"login": "comcxx11"}, "repo": {"name": "phpsysinfo/phpsysinfo"}, "created_at": "2015-01-01T15:00:05Z"}
{"id": "2489651128", "type": "ForkEvent", "actor": {"login": "Soufien"}, "repo": {"name": "wasabeef/awesome-android-libraries"}, "created_at": "2015-01-01T15:00:10Z"}
{"id": "2489651405", "type": "PushEvent", "actor": {"login": "hex7c0"}, "repo": {"name": "hex7c0/json-decrypt"}, "payload": {"commits": [{"author": {"name": "hex7c0"}, "message": "travis docker"}, {"author": {"name": "hex7c0"}, "message": "update devDependencies"}]}, "created_at": "2015-01-01T15:00:39Z"}
{"id": "2489651591", "type": "WatchEvent", "actor": {"login": "mhparker23"}, "repo": {"name": "vinta/awesome-python"}, "created_at": "2015-01-01T15:01:05Z"}
{"id": "2489661785", "type": "WatchEvent", "actor": {"login": "x2bool"}, "repo": {"name": "vinta/awesome-python"}, "created_at": "2015-01-01T15:23:34Z"}
"""

dbutils.fs.mkdirs(f"{BASE}/lessons")
for name, text in [("taxi12.csv", TAXI12_CSV), ("gh8.jsonl", GH8_JSONL)]:
    with open(f"{BASE}/lessons/{name}", "wb") as f:   # bytes: the same file on every computer
        f.write(text.encode())
    print("wrote", name, len(text.splitlines()), "lines")

## Step 6 · Final check

The first cell lists what is in each folder.
The second cell counts rows and files and compares them with the expected numbers.
Every line should start with `OK`.

In [ ]:
for folder in ["", "nyctaxi", "nyctaxi/green_2017", "gharchive", "gutenberg/1/1/1", "lessons"]:
    print(f"{BASE}/{folder}")
    for info in dbutils.fs.ls(f"{BASE}/{folder}"):
        print("   ", info.name, info.size)

In [ ]:
import glob

checks = {
    "taxi CSV rows":     (csv_trips.count(), 2705926),
    "taxi Parquet rows": (spark.read.parquet(f"{BASE}/nyctaxi/green_2017").count(), 2705926),
    "GitHub events":     (spark.read.json(f"{BASE}/gharchive/*.json.gz").count(), 35061),
    "book files":        (len(glob.glob(f"{BASE}/gutenberg/1/1/1/*/*/?????.txt")), 23),
    "lesson taxi rows":  (spark.read.csv(f"{BASE}/lessons/taxi12.csv", header=True).count(), 12),
    "lesson events":     (spark.read.json(f"{BASE}/lessons/gh8.jsonl").count(), 8),
}
for name, (got, expected) in checks.items():
    status = "OK   " if got == expected else "CHECK"
    print(f"{status} {name}: {got:,} (expected {expected:,})")

**All OK?** You are ready for the lessons. Keep `raw/`: it lets you rebuild everything by running this notebook again.

If a line says `CHECK`, run the notebook again from Step 2. Missing files are the usual cause.